In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Here we read in data showing the average annual temperature history at Portland airport.

In [ ]:
temp = pd.read_csv("ap_temp.csv")
pdx_temp = temp[temp["Airport"] == "PDX"]

In [ ]:
sns.histplot(x = "T", data = pdx_temp)

NameError: name 'sns' is not defined

#### Take one sample of the data

Consider a sample of 50% of the population (temperatures at PDX)

In [ ]:
one_sample = pdx_temp.sample(frac = 0.5, replace = False)
one_mean = np.mean(one_sample["T"])

one_mean

The resulting distribution might look more or less similar to the population's distribution depending on our random sample. Remember that normally **we don't know what the population distribution looks like**.

In [ ]:
sns.histplot(one_sample["T"])

If we're using the sample mean as an estimate of the population mean, how can confident can we be in our statistic?

#### Now take multiple samples and calculate the mean of each

In [ ]:
ten_means = np.empty(0)

# Take 10 means
for i in np.arange(1,10):
        samp_mean = np.mean(pdx_temp["T"].sample(frac = 0.5, replace = False))
        ten_means = np.append(ten_means, samp_mean)

sns.histplot(ten_means)

Now take progressively more samples and find the mean.

In [ ]:
many_means = np.empty(0)

# Try more and more sampling
for i in np.arange(1,50):
        samp_mean = np.mean(pdx_temp["T"].sample(frac = 0.5, replace = False))
        many_means = np.append(many_means, samp_mean)

sns.histplot(many_means)

This represents a possible sampling distribution of the mean for the population. As you can see, when we find the sample of the mean, **we can treat it as if it was randomly drawn from a normal distribution**.

If we took a single sample, we would expect the mean to fall somewhere in the distribution, with likelihood decreasing toward the edges of the distribution.

In [ ]:
one_sample = pdx_temp.sample(frac = 0.5, replace = False)
one_mean = np.mean(one_sample["T"])

sns.histplot(many_means)
plt.axvline(one_mean, c = "red")

### Bootstrapping

In practice, we can't take as many samples from the population as we want. Instead, we need to estimate the variability of possible potential sample estimates by resampling the original sample many times **i.e.** we need to bootstrap.

In [ ]:
sns.histplot(one_sample["T"])

In [ ]:
np.mean(one_sample["T"])

The Central Limit Theorem says that, given a large enough sample size, we can assume that the mean of our sample was drawn from a normal distribution of sample means. 

In [ ]:
boot = np.empty(0)

# Try different amounts of bootstraps until we have achieves an approximately normal sampling distribution
for i in np.arange(1,10000):
        samp_mean = np.mean(one_sample["T"].sample(frac = 1, replace = True))
        boot = np.append(boot, samp_mean)

sns.histplot(boot)

In [ ]:
#np.percentile(boot, 2.5)
np.percentile(boot, 97.5)

#### Visualize 95% confidence interval

To estimate confidence in the estimated mean, we create an interval representing ~2 standard deviations from the sampling distribution's mean.

In [ ]:
sns.histplot(boot)
plt.axvline(x = np.percentile(boot, 2.5), linestyle = "dashed", c= "red")
plt.axvline(x = np.percentile(boot, 97.5), linestyle = "dashed", c= "red")
#plt.scatter(np.array([np.percentile(boot, 2.5), np.percentile(boot, 97.5)]), np.array([0, 0]), color='red', lw=3, zorder=1, s = 200, marker = "^")

In [ ]:
np.mean(pdx_temp["T"])

Based on our sample, we would expect the true population mean to be somewhere in our 95% confidence interval.

### Comparison of two distributions

Has the temperature shifted with time in Portland?

Another way of phrasing this: can the difference in the distributions in temperature before and after 1987 be plausibly attributed to chance?

In [ ]:
sns.histplot(pdx_temp["T"][pdx_temp["Year"] < 1987])
sns.histplot(pdx_temp["T"][pdx_temp["Year"] >= 1987], color = "red")
plt.xlabel("Temperature (C)")

**Null hypothesis**: No difference in means

**Alternative hypothesis**: Yes difference in means

In [ ]:
before_1987 = pdx_temp[(pdx_temp["Year"] > 1940) & (pdx_temp["Year"] < 1987)]
after_1987 = pdx_temp[pdx_temp["Year"] >= 1987]

before_1987_mean = before_1987["T"].mean()
after_1987_mean = after_1987["T"].mean()

In [ ]:
before_1987_mean

In [ ]:
after_1987_mean

Bootstrap to quantify confidence in the differences temperature before and after 1987

In [ ]:
boot_post1987 = np.empty(0)

for i in np.arange(1,10000):
        
        resamp = after_1987.sample(frac = 1, replace = True)
        
        boot_post1987 = np.append(boot_post1987, np.mean(resamp["T"]))

sns.histplot(boot_post1987)

In [ ]:
before_1987_mean

We we can compare our post 1987 confidence interval with the average for pre 1987

In [ ]:
sns.histplot(boot_post1987)

plt.axvline(x = np.percentile(boot_post1987, 97.5), c = "red", linestyle = "dashed")
plt.axvline(x = np.percentile(boot_post1987, 2.5), c = "red", linestyle = "dashed")

In [ ]:
sns.histplot(boot_post1987)

plt.axvline(x = np.percentile(boot_post1987, 99.5), c = "red", linestyle = "dashed")
plt.axvline(x = np.percentile(boot_post1987, .5), c = "red", linestyle = "dashed")

plt.axvline(x = 12.28, c = "orange", linestyle = "dashed")

If we consider both temperature measurements as samples, we should also explicitly consider the variability of each. 

In [ ]:
boot_pre1987 = np.empty(0)

for i in np.arange(1,10000):
        
        resamp = before_1987.sample(frac = 1, replace = True)
        
        boot_pre1987 = np.append(boot_pre1987, np.mean(resamp["T"]))

In [ ]:
sns.histplot(boot_pre1987)
plt.scatter(np.array([np.percentile(boot_pre1987, 2.5), np.percentile(boot_pre1987, 97.5)]), np.array([0, 0]), color = "black", lw=3, zorder=1, s = 200, marker = "^")

sns.histplot(boot_post1987, color = "red")
plt.scatter(np.array([np.percentile(boot_post1987, 2.5), np.percentile(boot_post1987, 97.5)]), np.array([0, 0]), color = "black", lw=3, zorder=1, s = 200, marker = "^")

In [ ]:
after_1987_mean - before_1987_mean